In [1]:
import glob, os, shutil
src = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/VERSION", recursive=True)
       if open(p).read().strip() == "v8.6"]
assert src, "v8.6 code not found: attach the dataset with unlearning_tofu_experiment_v8_6.zip"
dst = "/kaggle/working/unlearning-project"
shutil.rmtree(dst, ignore_errors=True); shutil.copytree(src[0], dst); os.chdir(dst)
for d in ("results/v6", "logs/v6"):
    shutil.rmtree(d, ignore_errors=True)        # never carry results of an attached notebook into this one
!pip install -q rouge-score bitsandbytes peft
!pip uninstall -y -q torchao
os.makedirs("results/v6", exist_ok=True)
!python env_capture.py > results/v6/environment_run25b2.txt
!nvidia-smi -L
print("code version:", open("VERSION").read().strip())


GPU 0: Tesla T4 (UUID: GPU-f28dc331-75a8-8c4d-c5cc-5aa8a4d84ee2)
GPU 1: Tesla T4 (UUID: GPU-fa3972e3-8460-2caa-2ffe-45f99ce7753d)
code version: v8.6


In [2]:
# part b needs the output of the run25a2 notebook (Add input -> Notebook output)
import glob
a = glob.glob("/kaggle/input/**/run25_ckpt/A_s2/lora_chain.json", recursive=True)
o = glob.glob("/kaggle/input/**/run25_s2/O_s2_raw.json", recursive=True)
print("Model A adapter:", a[:1]); print("oracle results:", o[:1])
assert a and o, "attach the output of the run25a2 notebook first"


Model A adapter: ['/kaggle/input/notebooks/aimanaslam120/run25a2-ipynb/run25_ckpt/A_s2/lora_chain.json']
oracle results: ['/kaggle/input/notebooks/aimanaslam120/run25a2-ipynb/unlearning-project/results/v6/run25_s2/O_s2_raw.json']


In [3]:
%cd /kaggle/working/unlearning-project
!CKPT_DIR=/tmp/ckpt RUN25_KEEP=/kaggle/working/run25_ckpt python experiments_v6.py run25b2


/kaggle/working
[15:31:44] === v8.6 RUN 25b2  GPUs=2  model=Qwen/Qwen2.5-3B-Instruct (LoRA)  replicate 2
[15:31:47] START  train_P10_s2  (GPU 0)
[16:01:16] DONE   train_P10_s2  (29.5 min)
[16:01:16] START  train_r25_npo_rt_forget10_lr0.0001_s2  (GPU 0)
[16:01:16] START  train_r25_anchor_p10_forget10_lr0.0003_s2  (GPU 1)
[18:07:34] DONE   train_r25_npo_rt_forget10_lr0.0001_s2  (126.3 min)
[18:07:34] START  eval_run25_s2_npo_rt_lr0.0001_ep10_s2  (GPU 0)
[18:22:36] DONE   train_r25_anchor_p10_forget10_lr0.0003_s2  (141.3 min)
[18:22:36] START  eval_run25_s2_anchor_p10_lr0.0003_ep10_s2  (GPU 1)
[18:56:19] DONE   eval_run25_s2_npo_rt_lr0.0001_ep10_s2  (48.7 min)
[18:56:19] START  eval_run25_s2_npo_rt_lr0.0001_ep5_s2  (GPU 0)
[19:03:04] DONE   eval_run25_s2_anchor_p10_lr0.0003_ep10_s2  (40.5 min)
[19:03:04] START  eval_run25_s2_anchor_p10_lr0.0003_ep5_s2  (GPU 1)
[19:41:18] DONE   eval_run25_s2_npo_rt_lr0.0001_ep5_s2  (45.0 min)
[19:43:26] DONE   eval_run25_s2_anchor_p10_lr0.0003_ep5_s2  (40

In [4]:
%cd /kaggle/working/unlearning-project
!zip -qr /kaggle/working/results_v8_6_run25b2.zip results/v6/run25_s2 results/v6/environment_run25b2.txt logs/v6
!ls -la /kaggle/working


/kaggle/working
total 560
drwxr-xr-x 3 root root   4096 Nov 17 19:43 .
drwxr-xr-x 8 root root   4096 Nov 17 15:30 ..
-rw-r--r-- 1 root root   8267 Nov 17 19:43 __notebook__.ipynb
-rw-r--r-- 1 root root 547575 Nov 17 19:43 results_v8_6_run25b2.zip
drwxrwxrwx 8 root root   4096 Nov 17 15:30 unlearning-project
